# Shakespeare SLM — end-to-end training on Colab

Trains a 172.6M-parameter transformer from scratch, then fine-tunes it on tiny
Shakespeare and ships it as a web app. Every model, tokenizer and training loop
in this project is hand-written; the only heavy dependency is PyTorch.

**Pipeline:** download corpora -> tokenize to `.bin` -> pretrain on FineWeb-Edu
-> fine-tune on Shakespeare -> export for inference -> publish -> serve.

**Runtime:** set this to an **A100 GPU** (Runtime -> Change runtime type) before
running anything past the setup section. Pretraining takes roughly 6-7 hours.

**Layout.** The repository lives on Google Drive so checkpoints survive a
disconnect, but the training data is copied to Colab's local disk each session
because the dataset is memory-mapped with random access and reading it through
the Drive FUSE layer would starve the GPU.

```
MyDrive/shakespeare-slm/
├── repo/     <- this git repo; checkpoints are written inside it
└── data/     <- the .bin token files
```

---
## 1. Setup

Mount Drive so the repo and checkpoints persist between sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

The repo is private, so cloning needs a GitHub token. Create a fine-grained
token with *Contents: read* and store it in Colab's **Secrets** panel (the key
icon) as `GH_TOKEN` — never paste a token into a cell, it gets saved with the
notebook.

In [ ]:
import os
from google.colab import userdata

os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['REPO'] = 'franciscomendez114/shakespeare-pretrain-finetune-slm'

REPO_DIR = '/content/drive/MyDrive/shakespeare-slm/repo'

Clone on the first run; afterwards just pull. Checkpoints and `.bin` files are
gitignored, so `git pull` updates code without touching them.

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone https://$GH_TOKEN@github.com/$REPO.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull

---
## 2. Data

Nothing in `data/` is committed — the corpora are 10+ GB. This section rebuilds
them from scratch, and only needs to run **once**; afterwards the `.bin` files
live in Drive and later sessions just copy them.

The work happens in a throwaway copy of the repo on local disk. `prepare_data.py`
writes relative to its own location, and the 10 GB of raw text would otherwise
land in Drive and eat the quota for no reason — local disk is faster and
disappears when the session ends.

In [ ]:
!cp -r $REPO_DIR /content/prep
!pip install -q datasets

Download ~11 GB of FineWeb-Edu (documents separated by `<|endoftext|>`) and the
tiny Shakespeare corpus. Both scripts write to a `.part` file and rename only on
success, so an interrupted download can never look complete.

In [ ]:
!cd /content/prep && python data/scripts/download_fineweb.py \
    --gb 11 --out data/raw/pretraining-fineweb.txt
!cd /content/prep && python data/scripts/download_shakespeare.py

Tokenize both corpora to flat `uint16` arrays with the project's own BPE
tokenizer (20k vocab, trained separately and committed under
`tokenizer/artifacts/`). Streams line by line, so the multi-GB file is never
loaded into memory. Expect ~12 minutes.

This yields ~3.73B FineWeb training tokens and 472k Shakespeare training tokens.

In [ ]:
!cd /content/prep && python data/scripts/prepare_data.py

Copy the token files to Drive so this never has to run again.

In [ ]:
!mkdir -p /content/drive/MyDrive/shakespeare-slm/data
!cp /content/prep/data/processed/*.bin /content/drive/MyDrive/shakespeare-slm/data/
!ls -lh /content/drive/MyDrive/shakespeare-slm/data/

---
## 3. Stage the data locally

**Run this at the start of every session.** `TokenDataset` memory-maps the
`.bin` files and reads random windows; served over Drive those reads are slow
enough to leave an A100 idle. Copying to local disk first is what keeps the GPU
fed.

In [ ]:
!mkdir -p /content/data
!cp /content/drive/MyDrive/shakespeare-slm/data/*.bin /content/data/
!nvidia-smi --query-gpu=name,memory.total --format=csv

---
## 4. Pretrain on FineWeb-Edu

13,150 steps at an effective batch of 256 sequences x 1024 tokens = **3.45B
tokens**, which is 20 tokens per parameter — the Chinchilla compute-optimal
ratio for a 172.6M model. Hyperparameters come from `configs/pretrain.yaml`.

The loop uses bf16 autocast, TF32, fused AdamW and `torch.compile` on CUDA, and
writes a full training checkpoint (model + optimizer + scheduler + step) every
1,000 steps. **If Colab disconnects, just re-run this cell** — it finds
`last.pt` and prints `resuming from ... at step N`.

One rule: do not change `MAX_TRAINING_STEPS` between sessions. The checkpoint
restores the scheduler's original `T_max`, and cosine annealing is periodic, so
the learning rate would climb back toward its peak instead of decaying.

Expect ~6-7 hours and a final validation loss near **1.79** (perplexity ~6.0).

In [ ]:
%cd $REPO_DIR
!DATA_DIR=/content/data python -m training.pretrain

---
## 5. Fine-tune on tiny Shakespeare

Starts from the pretrained weights — model weights only, not the optimizer
state, since fine-tuning runs its own learning-rate schedule at 1/10th the peak.

Only 150 steps here. The Shakespeare training split is 461 windows, so this is
about 5 epochs, and validation loss bottoms out around step 130 before
overfitting sets in. The trainer therefore keeps **`best.pt`** (lowest
validation loss) alongside `final.pt` (last step) — watch the `*best` marker in
the log to see where the true minimum lands.

Takes about a minute.

In [ ]:
%cd $REPO_DIR
!DATA_DIR=/content/data python -m training.finetune

---
## 6. Compare the two models

The clearest evidence fine-tuning worked: same prompt, same settings, only the
checkpoint differs. The pretrained model has never seen a play and treats
`ROMEO:` as ordinary web text; the fine-tuned one answers in play format.

Temperature matters more than you would expect. This model's distribution is
sharp enough that 0.8 samples nearly greedily and it falls into repetition
loops — 0.9-1.0 reads much better.

In [ ]:
%cd $REPO_DIR
!python scripts/generate_sample.py --checkpoint checkpoints/pretrain/final.pt \
    --prompt "ROMEO:" --tokens 200 --temperature 0.9 --top-k 100 --seed 0
!python scripts/generate_sample.py --checkpoint checkpoints/finetune/best.pt \
    --prompt "ROMEO:" --tokens 200 --temperature 0.9 --top-k 100 --seed 0

---
## 7. Export for inference

A training checkpoint is 2.07 GB, but two thirds of that is Adam optimizer state
that inference never touches. This strips it to weights only and casts to fp16,
giving a **345 MB** bundle: `model.pt`, a `config.json` describing the
architecture, and a copy of the tokenizer.

The bundle is self-contained — the app rebuilds the model from `config.json` and
never reads `configs/`. Both checkpoints are exported so the app can offer both.

Writing to `/content` rather than Drive: this is a build artifact headed for
Hugging Face, not something worth storing twice.

In [ ]:
%cd $REPO_DIR
!python scripts/export_model.py --checkpoint checkpoints/finetune/best.pt  --out /content/export
!python scripts/export_model.py --checkpoint checkpoints/pretrain/final.pt --out /content/export-pretrained

Sanity check before publishing: load the bundle exactly as the deployed app
will — nothing read from `configs/`, fp16 weights cast back to fp32 for CPU —
and generate.

In [ ]:
%cd $REPO_DIR
!python -c "
import torch
from inference.load_model import load_exported_model, load_exported_tokenizer
from inference.generate import generate
m = load_exported_model('/content/export', torch.device('cpu'))
t = load_exported_tokenizer('/content/export')
print(generate(m, t, 'ROMEO:', 120, 0.9, 100))"

---
## 8. Publish the weights

The models are far too large for GitHub (100 MB hard limit per file), so they
live on the Hugging Face Hub and the app downloads them at startup. Needs a
fine-grained **Write** token stored in Colab Secrets as `HF_TOKEN`.

Both bundles go into one repo as `finetuned/` and `pretrained/` subfolders,
which is the layout `app/app.py` expects.

In [ ]:
!pip install -q huggingface_hub

from google.colab import userdata
from huggingface_hub import HfApi

MODEL_REPO = 'bahamawama/shakespeare-slm'
api = HfApi(token=userdata.get('HF_TOKEN'))
api.create_repo(MODEL_REPO, repo_type='model', private=False, exist_ok=True)

api.upload_folder(folder_path='/content/export',
                  path_in_repo='finetuned',  repo_id=MODEL_REPO)
api.upload_folder(folder_path='/content/export-pretrained',
                  path_in_repo='pretrained', repo_id=MODEL_REPO)

print(f'https://huggingface.co/{MODEL_REPO}')

---
## 9. Run the app

A Gradio UI with a toggle between the two checkpoints, sliders for sampling
settings, and token-by-token streaming. It pulls the weights from the Hub repo
published above, so it needs nothing from Drive.

`share=True` prints a public `*.gradio.live` URL that stays live for 72 hours —
enough to demo it, but not a deployment. For something permanent the same
`app/app.py` runs on any host with ~2 GB of RAM.

In [ ]:
!pip install -q gradio

%cd $REPO_DIR
import app.app as shakespeare_app
shakespeare_app.demo.launch(share=True)